# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import pandas as pd
from pathlib import Path

# Try both possible dataset locations
paths = [
    "data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv"
]

for p in paths:
    if Path(p).exists():
        data_path = p
        break
else:
    raise FileNotFoundError("Dataset not found!")

print("Using:", data_path)

df = pd.read_csv(data_path)

print("\nShape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()


Using: ../../data/raw/content_refresh_anonymized.csv

Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Rule

I will prioritize content for refresh if it is old, has many impressions, low CTR, and traffic is declining.

The goal is to identify pages that still receive visibility but may perform better after being refreshed.

### Reason Codes

- STALE_CONTENT
- LOW_CTR
- HIGH_IMPRESSIONS
- DECLINING_TREND

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
import pandas as pd
from pathlib import Path

# Load dataset
paths = [
    "data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv"
]

for p in paths:
    if Path(p).exists():
        data_path = p
        break

df = pd.read_csv(data_path)

# ----------------------------
# Baseline scoring rule
# ----------------------------

score = (
    (df["days_since_last_update"] / df["days_since_last_update"].max()) * 40
    + (df["impressions_90d"] / df["impressions_90d"].max()) * 30
    + ((1 - df["ctr"]) * 20)
)

# declining pages get bonus
score += (df["trend_direction"] == "down") * 10

df["baseline_score"] = score

# Action label
df["action_label"] = "Refresh Content"

# Reason Code
df["reason_code"] = "STALE_CONTENT"

# Rank
df = df.sort_values("baseline_score", ascending=False)

# Save CSV
output_path = Path("../../work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)

print("CSV saved to:", output_path)

df.head(20)

CSV saved to: ..\..\work\outputs\baseline_action_score.csv


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,action_label,reason_code
26242,content_55a5b1c46474,client_4ec9599fc2,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.00,0.00,0.00,low,page_1,down,-88.5,70.002028,Refresh Content,STALE_CONTENT
29384,content_f6fdf87348f6,client_4ec9599fc2,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.00,0.00,0.00,low,page_3_5,down,-100.0,70.000116,Refresh Content,STALE_CONTENT
24216,content_1b4ec72dafd4,client_4ec9599fc2,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.00,50.00,0.00,low,page_1,down,-100.0,69.892877,Refresh Content,STALE_CONTENT
6653,content_5fe46e04994d,client_4e07408562,1900.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,4.23,26.90,0.38,excellent,page_1,down,-44.8,68.352815,Refresh Content,STALE_CONTENT
8631,content_e2b702f4f92b,client_4ec9599fc2,NaN,NaN,NaN,NaN,keyword article,NaN,1246.0,8740.0,...,0.00,50.00,0.00,low,page_1,down,-72.7,65.819433,Refresh Content,STALE_CONTENT
21984,content_02b0d6e30129,client_19581e27de,110.0,0.40,MEDIUM,0.59,keyword article,transactional,NaN,NaN,...,0.00,0.00,0.00,low,page_1,down,-95.6,63.575882,Refresh Content,STALE_CONTENT
7509,content_7a888d3d99c8,client_19581e27de,90.0,0.46,MEDIUM,0.72,keyword article,transactional,NaN,NaN,...,0.00,0.00,0.00,low,deep,down,-100.0,63.571189,Refresh Content,STALE_CONTENT
18841,content_94991fe6268c,client_19581e27de,10.0,0.32,LOW,0.12,keyword article,commercial,NaN,NaN,...,0.00,0.00,0.00,low,striking,down,-100.0,63.566089,Refresh Content,STALE_CONTENT
3723,content_f488400fca67,client_9400f1b21c,NaN,NaN,NaN,NaN,keyword article,NaN,1113.0,8048.0,...,0.00,0.00,0.00,low,page_1,down,-44.0,62.716757,Refresh Content,STALE_CONTENT
15882,content_129753e3095f,client_19581e27de,10.0,0.55,MEDIUM,0.04,keyword article,transactional,NaN,NaN,...,0.00,100.00,0.00,low,page_3_5,down,-50.0,62.708180,Refresh Content,STALE_CONTENT


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 Review

The highest ranked pages are prioritized because they are older, receive high impressions, have relatively low CTR, and show declining trends.

These pages are good candidates for content refresh.

Confidence: Medium

What could make these recommendations wrong?

- Seasonal traffic
- Recently updated content not reflected yet
- Temporary ranking fluctuations
- External events affecting search traffic

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some pages may receive high scores because of temporary traffic drops or seasonal behavior.

Pages with low impressions may not benefit from refresh.

### Leakage Check

No future information was used.

No product labels or future performance metrics were included.

Only current historical features were used for scoring.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

[x]